##Training Summarization Model Using PEFT technique

In [ ]:
!pip install transformers evaluate transformers[torch]
!pip install py7zr # for samsum dataset
!pip install -U datasets
!pip install peft

In [ ]:
# Load model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("HFS26/bart-cnn-samsum-finetune-model")
model = AutoModelForSeq2SeqLM.from_pretrained("HFS26/bart-cnn-samsum-finetune-model")

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ingeniumacademy/samsum")
dataset

In [ ]:
#prepare dataset
def tokenize_inputs(example):
    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt = "\n\nSummary: "
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example['dialogue']]
    tokenized_prompt = tokenizer(prompt, padding='max_length', truncation=True, return_tensors='pt', max_length=512)
    tokenized_summary = tokenizer(example['summary'], padding='max_length', truncation=True, return_tensors='pt', max_length=512)

    example['input_ids'] = tokenized_prompt['input_ids']
    example['attention_mask'] = tokenized_prompt['attention_mask']
    example['labels'] = tokenized_summary['input_ids']

    return example

tokenizer.pad_token = tokenizer.eos_token
tokenized_datasets = dataset.map(tokenize_inputs, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'dialogue', 'summary'])
tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

In [ ]:
print("print tokenized datasets : ")
print(tokenized_datasets['train'].shape)
print(tokenized_datasets['validation'].shape)
print(tokenized_datasets['test'].shape)

### Create PEFT Model using LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    r=32, # 8, 16, 32
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM
)

In [ ]:
peft_model = get_peft_model(model, peft_config=lora_config)

In [ ]:
# login huggig face hub
from huggingface_hub import notebook_login
notebook_login()

###Train PEFT Model

In [ ]:
from transformers import TrainingArguments, Trainer

peft_training_args = TrainingArguments(
    output_dir="./bart-cnn-samsum-peft",  # local directory
    hub_model_id="HFS26/bart-cnn-samsum-peft-trained",  # identifier on the Hub
    learning_rate=1e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    auto_find_batch_size=True,
    eval_strategy='epoch',
    logging_steps=10
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

In [ ]:
peft_model.print_trainable_parameters()

In [ ]:
peft_trainer.train()

In [ ]:
# save PEFT trainer to HF hub
peft_trainer.push_to_hub()

In [ ]:
def generate_summary(input, llm):
  input_prompt = f"""
                  Summarize the following conversation, {sample} Summary:
                  """

  input_ids = tokenizer(sample, return_tensors='pt')
  tokenized_output = llm.generate(input_ids=input_ids['input_ids'], min_length=30, max_length=200)
  output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)

  return output

### Reload & Test

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("HFS26/bart-cnn-samsum-finetune-model")

peft_model_base = AutoModelForSeq2SeqLM.from_pretrained("HFS26/bart-cnn-samsum-finetune-model")

loaded_peft_model = PeftModel.from_pretrained(
    peft_model_base,
    "HFS26/bart-cnn-samsum-peft-trained",
    is_trainable=False
)

In [ ]:
sample = dataset['test'][0]['dialogue']
label = dataset['test'][0]['summary']

output = generate_summary(sample, llm=loaded_peft_model)
print("Sample : \n")
print(sample)
print("-------------------")
print("Summary : \n")
print(output)
print("Label Summary:\n")
print(label)